# Building the Pipeline

In [1]:
import pyspark
from pyspark.sql import SparkSession
import logging
import os
import sys
import traceback
import numpy as np
import datetime

from Batch_ETL_Functions import *

In [2]:
Session_Name = "Covid_example_ETL_pipeline"
Testing = False

In [4]:
if __name__ == "__main__":
    logger = initialize_logger(print_output = True)
    start = datetime.datetime.now()
    ETL_covid(logger, Session_Name, Caching = False)
    caching_time1 = datetime.datetime.now() - start

    if Testing:
        ### How much does Caching the Datfrane before the Transformations affect the Speed of the Algorithm?
        start = datetime.datetime.now()
        ETL_covid(logger, Session_Name, data_schema, Caching = True)
        caching_time2 = datetime.datetime.now() - start

        print(f"Difference Not_Caching - Caching (Before Transformation Steps): {caching_time1 - caching_time2}")

Spark Session created
Successfully read BatchData/Raw/cases.csv
Successfully read BatchData/Raw/deaths.csv
Successfully read BatchData/Raw/recoveries.csv
All data succesfully read.
root
 |-- Country: string (nullable = true)
 |-- Date: string (nullable = false)
 |-- Cases: integer (nullable = true)
 |-- Deaths: integer (nullable = true)
 |-- Recoveries: integer (nullable = true)

Successfully combined datasets
Number of partitions by default 1
Starting transformation ...
Successfully transformed column: 'Country
Date column successfully transformed from string into date format
Successfully transformed Case and Deaths columns.
Successfully added Mortality_Rate.
Successfully added Recovery_Rate.
Succesfully added Mortality_Rate_Change and Recovery_ChangeRate
85_064 entries have been processed. Of these, 0 failed inspection.
Successfull tranformation of data
root
 |-- Country: string (nullable = true)
 |-- Date: date (nullable = false)
 |-- Cases: integer (nullable = true)
 |-- Deaths: in

In [27]:
logger = initialize_logger()
spark = start_session(logger, "Test")

test = True
base_path = "BatchData/Test/Covid/Load_Data_Start" if test else "BatchData/Raw"
# Read in the singular files
confirmed_infections = extract_one_dataset(spark, f"{base_path}/cases.csv", logger, inferSchema = True)
confirmed_deaths = extract_one_dataset(spark, f"{base_path}/deaths.csv", logger, inferSchema = True)
confirmed_recoveries = extract_one_dataset(spark, f"{base_path}/recoveries.csv", logger, inferSchema = True)

#### unpivot the dataframes
confirmed_infections = confirmed_infections.unpivot(ids = "Country", values = confirmed_infections.columns[1:], variableColumnName = "Date", valueColumnName = "Cases")
confirmed_deaths = confirmed_deaths.unpivot(ids = "Country", values = confirmed_deaths.columns[1:], variableColumnName = "Date", valueColumnName = "Deaths")
confirmed_recoveries = confirmed_recoveries.unpivot(ids = "Country", values = confirmed_recoveries.columns[1:], variableColumnName = "Date", valueColumnName = "Recoveries")
# Combine thet files
combined = confirmed_infections.join(confirmed_deaths, on = ["Country", "Date"]).join(confirmed_recoveries, on = ["Country", "Date"])

combined.printSchema()


df  = Transform(combined, logger)

Spark Session created
Successfully read BatchData/Test/Covid/Load_Data_Start/cases.csv
Successfully read BatchData/Test/Covid/Load_Data_Start/deaths.csv
Successfully read BatchData/Test/Covid/Load_Data_Start/recoveries.csv
root
 |-- Country: string (nullable = true)
 |-- Date: string (nullable = false)
 |-- Cases: integer (nullable = true)
 |-- Deaths: integer (nullable = true)
 |-- Recoveries: integer (nullable = true)

Starting transformation ...
Successfully transformed column: 'Country
Date column successfully transformed from string into date format
Successfully transformed Case and Deaths columns.
Successfully added Mortality_Rate.
Successfully added Recovery_Rate.
Succesfully added Mortality_Rate_Change and Recovery_ChangeRate
85_064 entries have been processed. Of these, 0 failed inspection.
Successfull tranformation of data
root
 |-- Country: string (nullable = true)
 |-- Date: date (nullable = false)
 |-- Cases: integer (nullable = true)
 |-- Deaths: integer (nullable = true)

In [28]:
df.show()

+-----------+----------+-----+------+----------+------------+--------------------+--------------------+---------------------+--------------------+
|    Country|      Date|Cases|Deaths|Recoveries|SANITY_CHECK|      Mortality_Rate|       Recovery_Rate|Mortality_Rate_Change|Recovery_Rate_Change|
+-----------+----------+-----+------+----------+------------+--------------------+--------------------+---------------------+--------------------+
|AFGHANISTAN|2020-03-23|   41|     1|         1|        true|0.024390243902439025|0.024390243902439025|  -0.1707317073170731| -0.1707317073170731|
|AFGHANISTAN|2020-03-24|   43|     1|         1|        true|0.023255813953488372|0.023255813953488372| -0.04651162790697683|-0.04651162790697683|
|AFGHANISTAN|2020-03-25|   76|     2|         2|        true| 0.02631578947368421| 0.02631578947368421|  0.13157894736842102| 0.13157894736842102|
|AFGHANISTAN|2020-03-26|   80|     3|         2|        true|              0.0375|               0.025|  0.42500000000

In [29]:
combined.show()

+-------------+----------+-----+------+----------+
|      Country|      Date|Cases|Deaths|Recoveries|
+-------------+----------+-----+------+----------+
|United States|2020-01-22|    1|     0|         0|
|United States|2020-01-23|    1|     0|         0|
|United States|2020-01-24|    2|     0|         0|
|United States|2020-01-25|    2|     0|         0|
|United States|2020-01-26|    5|     0|         0|
|United States|2020-01-27|    5|     0|         0|
|United States|2020-01-28|    5|     0|         0|
|United States|2020-01-29|    5|     0|         0|
|United States|2020-01-30|    5|     0|         0|
|United States|2020-01-31|    6|     0|         0|
|United States|2020-02-01|    8|     0|         0|
|United States|2020-02-02|    8|     0|         0|
|United States|2020-02-03|   11|     0|         0|
|United States|2020-02-04|   11|     0|         0|
|United States|2020-02-05|   12|     0|         0|
|United States|2020-02-06|   12|     0|         0|
|United States|2020-02-07|   12

In [30]:
combined.write.mode("overwrite").csv("BatchData/Test/Covid/Load_Data_End", header = True)

In [31]:
df.write.mode("overwrite").csv("BatchData/Test/Covid/Transform_Test", header = True)